# 05 - Tree ensembles

In [ ]:
import sys
sys.path.append("..")

import numpy as np

from experiments.data import load_data, SEED, CATEGORICAL
from experiments.evaluation import select_candidate, record_result, diagnostics, comparison_table, inverse_log
from experiments.estimators import structured_preprocessor, native_features, catboost_features

In [ ]:
parts, manifest = load_data("data/processed/model_input.parquet")
train, val, test = parts["train"], parts["validation"], parts["test"]

y_train, y_val = train.price.to_numpy(), val.price.to_numpy()
log_y_train, log_y_val = np.log1p(y_train), np.log1p(y_val)

manifest["split_counts"]

## Helper

`run` fits each set of parameters, keeps the one with the lowest validation MAE, then scores it on the test set and saves the results to `reports/experiments/`.

In [ ]:
FORMAT = {"MAE": "{:,.0f}", "RMSE": "{:,.0f}", "R2": "{:.4f}", "MAPE": "{:.2f}"}


def run(key, candidates, fit, predict, X_val, X_test, extra=None):
    model, params, tuning = select_candidate(candidates, fit, y_val, lambda m: predict(m, X_val))
    display(tuning)
    print("Best params:", params)

    val_pred = predict(model, X_val)
    test_pred = predict(model, X_test)
    _, scores = record_result(key, manifest, parts, val_pred, test_pred, params, tuning, extra=extra)
    display(scores.style.format(FORMAT))
    diagnostics(key, test.price, test_pred)
    return model

## Random Forest

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.ensemble import RandomForestRegressor


def fit_forest(params):
    forest = RandomForestRegressor(**params, random_state=SEED, n_jobs=2)
    model = make_pipeline(structured_preprocessor(scale=False), forest)
    return model.fit(train, log_y_train)


def predict_forest(model, X):
    return inverse_log(model.predict(X), y_train)


forest_params = [
    {"n_estimators": 80, "max_depth": 18, "min_samples_split": 6, "min_samples_leaf": 3, "max_features": 0.5},
    {"n_estimators": 120, "max_depth": 24, "min_samples_split": 10, "min_samples_leaf": 5, "max_features": 1.0},
]
forest = run("forest", forest_params, fit_forest, predict_forest, val, test)

## LightGBM

Categorical columns use the pandas `category` type. Training stops early when the validation score stops improving.

In [ ]:
from lightgbm import LGBMRegressor, early_stopping

X_train_lgb, levels = native_features(train)
X_val_lgb, _ = native_features(val, levels)
X_test_lgb, _ = native_features(test, levels)


def fit_lgb(params):
    model = LGBMRegressor(**params, random_state=SEED, n_jobs=2, deterministic=True,
                          force_col_wise=True, verbosity=-1)
    model.fit(X_train_lgb, log_y_train, eval_set=[(X_val_lgb, log_y_val)],
              callbacks=[early_stopping(50, verbose=False)])
    return model


def predict_lgb(model, X):
    return inverse_log(model.predict(X), y_train)


lgb_params = [
    {"n_estimators": 1500, "learning_rate": 0.04, "num_leaves": 31, "max_depth": -1, "min_child_samples": 40,
     "reg_alpha": 0.1, "reg_lambda": 1.0},
    {"n_estimators": 1500, "learning_rate": 0.04, "num_leaves": 63, "max_depth": 10, "min_child_samples": 60,
     "reg_alpha": 1.0, "reg_lambda": 5.0},
]
lgb = run("lightgbm", lgb_params, fit_lgb, predict_lgb, X_val_lgb, X_test_lgb)

## CatBoost

In [ ]:
from catboost import CatBoostRegressor

X_train_cat = catboost_features(train)
X_val_cat = catboost_features(val)
X_test_cat = catboost_features(test)


def fit_cat(params):
    model = CatBoostRegressor(**params, cat_features=CATEGORICAL, random_seed=SEED,
                              thread_count=2, verbose=False, allow_writing_files=False)
    model.fit(X_train_cat, log_y_train, eval_set=(X_val_cat, log_y_val), early_stopping_rounds=60)
    return model


def predict_cat(model, X):
    return inverse_log(model.predict(X), y_train)


cat_base = {"iterations": 400, "learning_rate": 0.05, "max_ctr_complexity": 1,
            "border_count": 64, "counter_calc_method": "SkipTest"}
cat_params = [
    {**cat_base, "depth": 4, "l2_leaf_reg": 3.0},
    {**cat_base, "depth": 6, "l2_leaf_reg": 7.0},
]
cat = run("catboost", cat_params, fit_cat, predict_cat, X_val_cat, X_test_cat)

## Compare models

In [ ]:
comparison_table(manifest).style.format(FORMAT)